# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list, system_prompt
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mol_instruction_dataset = datasets.load_dataset(
    "zjunlp/Mol-Instructions",
    "Molecule-oriented Instructions",
    trust_remote_code=True,
)
qm9_data = mol_instruction_dataset['property_prediction']
qm9_homo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo
            )

In [3]:
def get_qm9_mol_list(
        qm9_data,
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(qm9_data)))
    for i in iter_bar:
        data_instance = qm9_data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError
    return list_tr_mol, list_te_mol

In [4]:
list_qm9_homo_tr_mol, list_qm9_homo_te_mol = get_qm9_mol_list(
    qm9_data=qm9_homo_data,
)

100%|██████████| 120746/120746 [00:45<00:00, 2674.08it/s]


In [5]:
import deepchem as dc
loading_fn = dc.molnet.load_qm9

tasks, datasets, transformers = loading_fn(
    featurizer="Raw",
    splitter="scaffold",
    reload=True,
)

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


In [6]:
datasets[0].X.shape

(105984,)

In [7]:

concat_X = np.concatenate(
    [datasets[0].X, datasets[1].X, datasets[2].X], axis=0
)
concat_y = np.concatenate(
    [datasets[0].y, datasets[1].y, datasets[2].y], axis=0
)

In [8]:
concat_X.shape

(132480,)

In [9]:
# get inchi
list_concat_X_inchi = [Chem.MolToInchi(mol) for mol in concat_X]

[15:19:09] WARNING: Accepted unusual valence(s): C(2)

[15:19:09] WARNING: Accepted unusual valence(s): C(2); Proton(s) added/removed

[15:19:09] WARNING: Omitted undefined stereo; Ambiguous stereo: center(s)

[15:19:09] WARNING: Omitted undefined stereo; Ambiguous stereo: center(s)

[15:19:09] WARNING: Proton(s) added/removed

[15:19:09] WARNING: Proton(s) added/removed

[15:19:09] WARNING: Proton(s) added/removed

[15:19:09] WARNING: Proton(s) added/removed

[15:19:09] WARNING: Proton(s) added/removed; Omitted undefined stereo; Ambiguous stereo: center(s)

[15:19:09] WARNING: Omitted undefined stereo; Ambiguous stereo: center(s)

[15:19:09] WARNING: Omitted undefined stereo; Ambiguous stereo: center(s)

[15:19:09] WARNING: Proton(s) added/removed

[15:19:09] WARNING: Proton(s) added/removed

[15:19:09] WARNING: Omitted undefined stereo; Ambiguous stereo: center(s)

[15:19:09] WARNING: Omitted undefined stereo; Ambiguous stereo: center(s)

[15:19:09] WARNING: Omitted undefined stereo;

In [18]:
list_qm9_te_inchi = [Chem.MolToInchi(mol) for mol in list_qm9_homo_te_mol]

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefined stereo

[15:26:42] WARNING: Omitted undefi

In [19]:
iter_bar = tqdm(range(len(list_concat_X_inchi)))
qm9_idxs = []
for i in iter_bar:
    if list_concat_X_inchi[i] not in list_qm9_te_inchi:
        qm9_idxs.append(i)

  0%|          | 0/132480 [00:00<?, ?it/s]

100%|██████████| 132480/132480 [00:00<00:00, 133334.88it/s]


In [21]:
len(list_qm9_te_inchi)

684

In [22]:
len(list_concat_X_inchi)

132480

In [27]:

list_tr_mols = concat_X[qm9_idxs]
# get list
list_tr_mols = list_tr_mols.tolist()

In [29]:
tasks # use 0, 1, 5, 6, 7, 9, 10, 11

['mu',
 'alpha',
 'homo',
 'lumo',
 'gap',
 'r2',
 'zpve',
 'cv',
 'u0',
 'u298',
 'h298',
 'g298']

In [23]:
len(qm9_idxs)

132302

In [42]:
subtask_full_name_dict = {
    "mu": "dipole_moment",
    "alpha": "isotropic_polarizability",
    "r2": "electronic_spatial_extent",
    "zpve": "zero_point_vibrational_energy",
    "cv": "heat_capacity_298K",
    "u298": "internal_energy_298K",
    "h298": "enthalpy_298K",
    "g298": "free_energy_298K",
}

In [41]:
subtask_full_name_dict[tasks[0]]

'dipole_moment'

In [39]:
tasks[subtask_idx]

'mu'

In [40]:
subtask_full_name_dict['mu']

'dipole_moment'

In [52]:
len(list_tr_data)

132302

In [ ]:
import datasets

for subtask_idx in [0, 1, 5, 6, 7, 9, 10, 11]:
    task = subtask_full_name_dict[tasks[subtask_idx]]
    task = f'qm9_{task}'
    instruction_templates = getattr(instructions_smol, task)

    list_mols = list_tr_mols
    list_labels = concat_y[:, subtask_idx].tolist()

    list_tr_data = get_data_list(
            list_mol=list_mols,
            list_label=list_labels,
            task=task,
            instruction_templates=instruction_templates)

    dataset = datasets.Dataset.from_list(list_tr_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_{task}_0219"
        )


  0%|          | 0/132302 [00:00<?, ?it/s]

 30%|███       | 40074/132302 [00:41<01:24, 1091.96it/s]